#### 导入必要的库

In [1]:
import torch
import sys
import numpy as np
sys.path.append("/mnt/data1/tyl/UserID/")

from dataloader import GetLoaderM3CV
from dataset.instance import set_seed
from models.best2models import  FBMSTSNet


/home/hustmx709/.conda/envs/tyl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


#### 加载数据集和模型

In [10]:
# P300, Rest 两个范式 Smooth Label t-SNE 可视化
# M3CV_Transient, M3CV_Motor ,M3CV_Steady, M3CV_SSVEP_SA, M3CV_P300, M3CV_Rest
split = 'M3CV_Transient'
s1_seed = 2024
t1_seed = 2028
gpuid = 6
kernels = [11, 21, 31, 41, 51]
fs = 250
Chans = 64
nbands = 12
nclass = 20
device = torch.device(f'cuda:{gpuid}' if torch.cuda.is_available() else 'cpu')
# 超参数
# P300: 2025/2026, Rest: 2026/2024, Transient: 2024/2028
s1_path = f'/mnt/data1/tyl/UserID/ModelSave/{split}/base_FBMSTSNet_SF4_{s1_seed}.pth'
t1_path = f'/mnt/data1/tyl/UserID/ModelSave/{split}/CELabelSmooth_FBMSTSNet_{t1_seed}.pth'

print("#==================================================>>>")
print(f"Before Smoothing, Seed: {s1_seed}, split: {split}")
# load models and weights
s1_model = FBMSTSNet(kernels=kernels, fs=fs, in_channels=Chans, nbands=nbands, num_classes=nclass).to(device)
s1_model.load_state_dict(torch.load(s1_path))
# set random seed
set_seed(s1_seed)
# load dataloaders
s1_trainloadr, s1_valloader, s1_testloader = GetLoaderM3CV(seed=s1_seed, split=split)
s1_feas, s1_labels, s1_preds = [], [], []
for i, (x,y) in enumerate(s1_testloader):
    s1_model.eval()
    x, y = x.to(device), y.to(device)
    with torch.no_grad():
        fea, logit = s1_model(x)
        pred_y = torch.argmax(logit, dim=1)
        s1_feas.append(fea.cpu().detach().numpy())
        s1_labels.append(y.cpu().detach().numpy())
        s1_preds.append(pred_y.cpu().detach().numpy())
[s1_feas, s1_labels, s1_preds] = [np.concatenate(i, axis=0) for i in [s1_feas, s1_labels, s1_preds]]
print(f"Before Smoothing, Acc: {float((s1_preds==s1_labels).sum()/len(s1_labels)) * 100 : .2f}%")
print("#==========>>>")

print("#==================================================>>>")
print(f"After Smoothing, Seed: {t1_seed}, split: {split}")
# load models and weights
t1_model = FBMSTSNet(kernels=kernels, fs=fs, in_channels=Chans, nbands=nbands, num_classes=nclass).to(device)
t1_model.load_state_dict(torch.load(t1_path))
# set random seed
set_seed(t1_seed)
# load dataloaders
t1_trainloadr, t1_valloader, t1_testloader = GetLoaderM3CV(seed=t1_seed, split=split)
t1_feas, t1_labels, t1_preds = [], [], []
for i, (x,y) in enumerate(t1_testloader):
    t1_model.eval()
    x, y = x.to(device), y.to(device)
    with torch.no_grad():
        fea, logit = t1_model(x)
        pred_y = torch.argmax(logit, dim=1)
        t1_feas.append(fea.cpu().detach().numpy())
        t1_labels.append(y.cpu().detach().numpy())
        t1_preds.append(pred_y.cpu().detach().numpy())
[t1_feas, t1_labels, t1_preds] = [np.concatenate(i, axis=0) for i in [t1_feas, t1_labels, t1_preds]]
print(f"After Smoothing, Acc: {float((t1_preds==t1_labels).sum()/len(t1_labels)) * 100 : .2f}%")
print("#==========>>>")

save_dict = {'s_feas': s1_feas, 't_feas': t1_feas, 's_labels': s1_labels, 't_labels': t1_labels}

np.save(f'./tsne/{split}_s{s1_seed}_t{t1_seed}.npy', save_dict)




#==================================================>>>
Before Smoothing, Seed: 2024, split: M3CV_Transient
数据比例-----训练集:验证集:测试集 = (2872, 1, 64, 1000):(718, 1, 64, 1000):(1791, 1, 64, 1000)
Before Smoothing, Acc:  66.72%
#==========>>>
#==================================================>>>
After Smoothing, Seed: 2028, split: M3CV_Transient
数据比例-----训练集:验证集:测试集 = (2872, 1, 64, 1000):(718, 1, 64, 1000):(1791, 1, 64, 1000)
After Smoothing, Acc:  81.57%
#==========>>>


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn import datasets
from sklearn.manifold import TSNE

class FeatureVisualize(object):
    '''
    Visualize features by TSNE
    '''

    def __init__(self, features, labels):
        '''
        features: (m,n)
        labels: (m,)
        '''
        self.features = features
        self.labels = labels

    def plot_tsne(self, seed=2024, perplexity=30, n_iter=1000):
        ''' Plot TSNE figure. Set save_eps=True if you want to save a .eps file.
        '''
        tsne = TSNE(n_components=2, init='pca', random_state=seed, perplexity=perplexity, n_iter=n_iter)
        features = tsne.fit_transform(self.features)
        x_min, x_max = np.min(features, 0), np.max(features, 0)
        data = (features - x_min) / (x_max - x_min)
        del features
        return data

#### 可视化

In [4]:


fig, axes = plt.subplots(1, 2, figsize=(12, 12), dpi=600, sharex=True, sharey=True)
fig.set_figwidth(12)
fig.set_figheight(12)

# before label smoothing
p300_remove_list = [0, 2, 3, 5, 6, 8, 11, 12, 13, 14, 15, 18]
choice = np.where(~np.isin(s1_labels, p300_remove_list))[0]
s1_fea = s1_feas[choice]
s1_label = s1_labels[choice]
t1_fea = t1_feas[choice]
t1_label = t1_labels[choice]
print("# p300 paradigm>>>>>>>>>>")
print("features shape:", s1_fea.shape)
print("labels shape:", s1_label.shape)

s1_vis = FeatureVisualize(s1_fea, s1_label)
data11 = s1_vis.plot_tsne(seed=s1_seed)
t1_vis = FeatureVisualize(t1_fea, t1_label)
data12 = t1_vis.plot_tsne(seed=t1_seed)

axes[0, 0].scatter(data11[:, 0], data11[:, 1], c=s1_label, cmap='rainbow', alpha=0.6, s=50)
axes[0, 0].set_title("P300 paradigm before label smoothing", fontsize=10)
axes[0, 0].set_aspect('equal')

axes[0, 1].scatter(data12[:, 0], data12[:, 1], c=t1_label, cmap='rainbow', alpha=0.6, s=50)
axes[0, 1].set_title("P300 paradigm after label smoothing", fontsize=10)
axes[0, 1].set_aspect('equal')

plt.subplots_adjust(wspace=0.3)  # 可以根据需要调整这个值
plt.tight_layout()
plt.show()

In [5]:
# # P300, Rest 两个范式 Smooth Label t-SNE 可视化
# splits = ['M3CV_P300', 'M3CV_Rest']
# split = splits[1]
# # 超参数
# # P300: 2025/2026, Rest: 2026/2024
# s2_seed = 2026
# t2_seed = 2024
# gpuid = 6
# kernels = [11, 21, 31, 41, 51]
# fs = 250
# Chans = 64
# nbands = 12
# nclass = 20
# device = torch.device(f'cuda:{gpuid}' if torch.cuda.is_available() else 'cpu')

# s2_path = f'/mnt/data1/tyl/UserID/ModelSave/{split}/base_FBMSTSNet_SF4_{s2_seed}.pth'
# t2_path = f'/mnt/data1/tyl/UserID/ModelSave/{split}/CELabelSmooth_FBMSTSNet_{t2_seed}.pth'

# print("#==================================================>>>")
# print(f"Before Smoothing, Seed: {s2_seed}, split: {split}")
# # load models and weights
# s2_model = FBMSTSNet(kernels=kernels, fs=fs, in_channels=Chans, nbands=nbands, num_classes=nclass).to(device)
# s2_model.load_state_dict(torch.load(s2_path))
# # set random seed
# set_seed(s2_seed)
# # load dataloaders
# s2_trainloadr, s2_valloader, s2_testloader = GetLoaderM3CV(seed=s2_seed, split=split)
# s2_feas, s2_labels, s2_preds = [], [], []
# for i, (x,y) in enumerate(s2_testloader):
#     s2_model.eval()
#     x, y = x.to(device), y.to(device)
#     with torch.no_grad():
#         fea, logit = s2_model(x)
#         pred_y = torch.argmax(logit, dim=1)
#         s2_feas.append(fea.cpu().detach().numpy())
#         s2_labels.append(y.cpu().detach().numpy())
#         s2_preds.append(pred_y.cpu().detach().numpy())
# [s2_feas, s2_labels, s2_preds] = [np.concatenate(i, axis=0) for i in [s2_feas, s2_labels, s2_preds]]
# print(f"Before Smoothing, Acc: {float((s2_preds==s2_labels).sum()/len(s2_labels)) * 100 : .2f}%")
# print("#==========>>>")

# print("#==================================================>>>")
# print(f"After Smoothing, Seed: {t2_seed}, split: {split}")
# # load models and weights
# t2_model = FBMSTSNet(kernels=kernels, fs=fs, in_channels=Chans, nbands=nbands, num_classes=nclass).to(device)
# t2_model.load_state_dict(torch.load(t2_path))
# # set random seed
# set_seed(t2_seed)
# # load dataloaders
# t2_trainloadr, t2_valloader, t2_testloader = GetLoaderM3CV(seed=t2_seed, split=split)
# t2_feas, t2_labels, t2_preds = [], [], []
# for i, (x,y) in enumerate(t2_testloader):
#     t2_model.eval()
#     x, y = x.to(device), y.to(device)
#     with torch.no_grad():
#         fea, logit = t2_model(x)
#         pred_y = torch.argmax(logit, dim=1)
#         t2_feas.append(fea.cpu().detach().numpy())
#         t2_labels.append(y.cpu().detach().numpy())
#         t2_preds.append(pred_y.cpu().detach().numpy())
# [t2_feas, t2_labels, t2_preds] = [np.concatenate(i, axis=0) for i in [t2_feas, t2_labels, t2_preds]]
# print(f"After Smoothing, Acc: {float((t2_preds==t2_labels).sum()/len(t2_labels)) * 100 : .2f}%")
# print("#==========>>>")

In [6]:
# import numpy as np
# import matplotlib.pyplot as plt
# from sklearn import datasets
# from sklearn.manifold import TSNE


# class FeatureVisualize1(object):
#     '''
#     Visualize features by TSNE
#     '''

#     def __init__(self, features, labels):
#         '''
#         features: (m,n)
#         labels: (m,)
#         '''
#         self.features = features
#         self.labels = labels

#     def plot_tsne(self,save_eps=False):
#         ''' Plot TSNE figure. Set save_eps=True if you want to save a .eps file.
#         '''
#         tsne = TSNE(n_components=2, init='pca', random_state=0)
#         features = tsne.fit_transform(self.features)
#         x_min, x_max = np.min(features, 0), np.max(features, 0)
#         data = (features - x_min) / (x_max - x_min)
#         del features
#         for i in range(data.shape[0]):
#             plt.text(data[i, 0], data[i, 1], str(self.labels[i]),
#                      color=plt.cm.Set1(self.labels[i] / 10.),
#                      fontdict={'weight': 'bold', 'size': 8})
#         plt.xticks([])
#         plt.yticks([])
#         plt.title('T-SNE')
#         if save_eps:
#             plt.savefig('tsne.eps', dpi=600, format='eps')
#         plt.show()

In [7]:
# # before label smoothing
# rest_remove_list = [0, 1, 2, 3, 4, 5, 6, 7, 8, 11,12, 15, 16, 17, 18, 19]
# choice1 = np.where(~np.isin(s2_labels, rest_remove_list))[0]
# # choice1 = np.random.choice(len(s2_labels), len(s2_labels), replace=False)
# s2_fea = s2_feas[choice1]
# s2_label = s2_labels[choice1]
# t2_fea = t2_feas[choice1]
# t2_label = t2_labels[choice1]
# print("# Rest paradigm>>>>>>>>>>")
# print("features shape:", s2_fea.shape)
# print("labels shape:", s2_label.shape)

# s2_vis = FeatureVisualize1(s2_fea, s2_label)
# s2_vis.plot_tsne()
# t2_vis = FeatureVisualize1(t2_fea, t2_label)
# t2_vis.plot_tsne()
